In [1]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-24 15:58:56--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8000::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-24 15:58:57 (14.7 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-06-24 15:58:57--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request s

In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
# Create an assitant:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. **Install Ollama** from [https://ollama.com/download](https://ollama.com/download) for your operating system:
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model** in a terminal:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. **Test that the local server is running**:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}
   ```

If you get a connection refused error while prompting, restart the Ollama server with:

```bash
!nohup ollama serve > nohup.out 2>&1 &
```


## Function Calling
In the previous lesson we built a RAG pipeline with `RAGBase.rag()`
and saw it fail on the "Olama" typo. The search returned nothing
useful, and the LLM had no way to recover.

The flow that broke:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    S[search - Olama - no useful results]
    A([LLM: I don't have information about Olama.])

    U --> S --> A
```

The pipeline is fixed: search, build prompt, LLM.

An agent puts the LLM in charge.

Instead of running search ourselves, we give the LLM a `search` tool.
It decides when to call it and what to search for.

The same typo question now goes like this:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

The LLM searched, saw the results were bad, and decided to try again
with a different query. It made that decision on its own. We didn't
write any code to handle typos.

The difference is about who makes the decisions:

- With RAG, the developer decides. We fix the steps up front, so
  search always runs once with the exact user query.
- With an agent, the LLM decides. It chooses which actions to take
  and when to stop.

The mechanism that makes this possible is function calling, and that's
what the rest of this lesson is about.

## Asking without tools

First, let's see what the LLM does without any tools. We ask it a
course-specific question and look at the answer.

```python
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text
```

The model answers from its general knowledge, something like "it
depends on the course" or "check the course website". It doesn't know
about our FAQ, so the answer is vague and not helpful. This is exactly
why we need RAG, and why we want to hand the model a tool.


## Defining the tool

First we define a top-level `search` function that queries the `index`
directly. The model will reference it by this name. We keep the Python
function and the tool name aligned so the dispatch is easier later.

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our
Python code, only a schema describing what the function does and what
arguments it takes. LLMs are language agnostic. At the end we're just
making an HTTP call, so we describe the tool in JSON rather than in
Python. The same schema would work from TypeScript or Java.

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

The `description` is the most important field, because the model reads
it to decide when to call the function. `parameters` is a JSON schema
for the arguments, and we mark `query` as required so the model always
fills it in.

## Sending the question with the tool

Now we send the same question as before, but this time we include the
tool in the request:

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [13]:
response.output.__len__()

1

In [14]:
call = response.output[0]

In [15]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course enrollment can I join late registration"}', call_id='call_CadIRSZ07lIVOzcsYXi8j3b0', name='search', type='function_call', id='fc_08a32f24fb25d068006a3d873409408197ba22cffd22e56f4c', namespace=None, status='completed')

In [16]:
import json
args =  json.loads(call.arguments)

In [18]:
results = search(**args)

In [ ]:
result_json = json.dumps(results, indent=2)

In [26]:
print(result_json)

[
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.",
    "doc_id": "74eb249bbf"
  },
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.",
    "doc_id": "69d122f12e"
  },
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questio

## Sending a call output
We are taking the last response to resend a call to get a new output

In [ ]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}


In [29]:
messages.append(call)

In [30]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course enrollment can I join late registration"}', call_id='call_CadIRSZ07lIVOzcsYXi8j3b0', name='search', type='function_call', id='fc_08a32f24fb25d068006a3d873409408197ba22cffd22e56f4c', namespace=None, status='completed')]

Next, we create a history with our messages adding the last response into 'messages'

In [31]:
messages.append(function_call_output)

In [32]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course enrollment can I join late registration"}', call_id='call_CadIRSZ07lIVOzcsYXi8j3b0', name='search', type='function_call', id='fc_08a32f24fb25d068006a3d873409408197ba22cffd22e56f4c', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_CadIRSZ07lIVOzcsYXi8j3b0',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "an

In [34]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [37]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open. If you’re just learning, you can start anytime.


## Token usage and cost
We just made two API calls instead of one. Each call we send to the
model costs money, so it's worth checking how much one tool-using turn
actually costs.

The response has a `usage` field with the token counts:

In [38]:
response.usage.input_tokens, response.usage.output_tokens

(776, 43)

**messages** | history:
1. make a call to the LLM <-- we pay
2. LLM decided to invoke search(' params')
3. We invoke the search, we have the results
4. Send the results back to the LLM (another call) <-- we pay again
5. LLM processes the results
6. LLM gives the answer
The cost of 2 requests are sended to the LLM

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools, the functions the agent can call to carry out the task. For
  us that's only `search`.
- Memory, the message history. We append every prompt, every model
  output, and every tool result. The agent reads this to know what it
  has already tried.

Everything below is the code that wires these three together inside a
loop.

## A function-call helper

We'll be running function calls repeatedly inside the loop, so let's
wrap that in a small helper. It turns the JSON arguments into a Python
dict, calls the right function, and serializes the result. We only
have one tool for now, so we dispatch on the function name directly.

In [48]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

## A developer prompt
So far we've relied on the model to figure out when to search. We make
that more reliable with a `developer` message that spells out how to
behave. This is where we give the agent its role. The same message
also pushes it toward multiple searches, so we get to watch the loop
run more than once.

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [44]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [46]:
response.output.__len__()

2

In [49]:
messages.extend(response.output)

for item in response.output:
    print(item)
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(call)
        messages.append(call_output)
    
    elif item.type == 'message':
        print("ASSISTANT:")
        print(item.content[0].text)

ResponseFunctionToolCall(arguments='{"query":"join course late enrollment discovered the course can I join"}', call_id='call_jYYYTqOQbK9DPoNCf78vANsZ', name='search', type='function_call', id='fc_0b02c1e316adff45006a3f083ecfe88190bd4bf2f3075b3f71', namespace=None, status='completed')
function_call: search {"query":"join course late enrollment discovered the course can I join"}
ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered course FAQ"}', call_id='call_4GBst7IO2qYrRipcF1joCnma', name='search', type='function_call', id='fc_0b02c1e316adff45006a3f083ecffc819089d969db48d89d6f', namespace=None, status='completed')
function_call: search {"query":"course enrollment late join discovered course FAQ"}


In [50]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course late enrollment discovered the course can I join"}', call_id='call_jYYYTqOQbK9DPoNCf78vANsZ', name='search', type='function_call', id='fc_0b02c1e316adff45006a3f083ecfe88190bd4bf2f3075b3f71', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered course FAQ"}', call_id='call_4GBst7IO2qYrRip

## The full agent loop

We wrap this in a `while` loop. The loop keeps calling the model until
it returns a response without any function calls. We also keep an
iteration counter so we can see how many round-trips happened.

In [52]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered late can I join enrollment late registration FAQ"}
function_call: search {"query":"course discovered just now can I still join FAQ"}
function_call: search {"query":"late enrollment join the course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. Also, you can usually start learning and submitting homework even if you didn’t register beforehand, as long as the forms are open.

If you want, I can also help you figure out the best way to start or what to do first.


## Encouraging multiple searches

There's a subtle issue here. The model often answers after the first
search, even when more searches would help. It reasons that it already
knows enough, so why bother. We push it to explore more by rewriting
the instructions.

In [55]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [56]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered the course can I join enrollment late registration FAQ"}
iteration #2...
function_call: search {"query":"certificate live cohort project submission while form is open peer review accepting submissions FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. Also, if you’re aiming for the certificate, you need to participate while the course is running live, since peer review is required.

If you want, I can also help you figure out the best way to start from here. Any other areas you want to explore?


Every framework that works with multiple calls or agents, need a set of instructions and history to make a loop into a call and reuse. Having an agentic-tool-call-loop. Example like Langchaing or Pydantic

## Wrapping it in a function

Let's wrap the loop in a function so we can reuse it. The function
takes the instructions and the question as parameters, and returns
the final answer.

In [57]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [58]:
agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while accepting submissions peer-review project while course running self-paced mode no certificate FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. The self-paced mode doesn’t qualify for a certificate.

If you want, I can also explain how to get started and what the weekly workflow looks like.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. The self-paced mode doesn’t qualify for a certificate.\n\nIf you want, I can also explain how to get started and what the weekly workflow looks like.'

<h3>!!! Note: *</h3>
Check workshop to avoid ilusions response or inputs not related to the data.<br>
  - [Workshop AI-SHIPING]:(https://aishippinglabs.com/workshops/agent-with-guardrails)

The handwritten agent loop from the previous lesson is educational but
repetitive. Every time you build a new agent, you'd write the same
while-loop, the same function-call handling, the same message
management.


## ToyAIKit
ToyAIKit wraps this pattern so you can focus on tools, prompts, and
behavior. We built it together in a DataTalks.Club workshop a while
back. It does the same thing as our handwritten loop with less
boilerplate. If you open its `runners` code, you'll find the same
`while True` loop we wrote by hand.

I use it here on purpose, because I don't want to pick a winner among
the production frameworks. ToyAIKit is small and easy to read, so when
something breaks you can see exactly what happened. That makes it handy
for developing and debugging locally before you go to production.

One caveat. ToyAIKit is a teaching and experimentation library, and it
is NOT meant for production use. We use it because it's minimal and you
can see what it does.

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [67]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [65]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [61]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

<function __main__.search(query)>

In [68]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [69]:
result.cost

CostInfo(input_cost=Decimal('0.00268875'), output_cost=Decimal('0.0015525'), total_cost=Decimal('0.00424125'))

In [70]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local installation setup"}', call_id='call_tLqA2xYXrolHVvnGEjVJxhQH', name='search', type='function_call', id='fc_0d592bafe9552966006a3f5fcb66988195b9a0ac4f5c94a465', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_tLqA2xYXrolHVvnGEjVJxhQH',
  'output': '[\n  {\n    "course": 

In [71]:
result2 = runner.loop(
    prompt="How do I run Olama locally?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


## Interactive chat

For a chat-like workflow, run the built-in input loop:


In [74]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run model in docker ?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"run model in docker how to run model in docker course FAQ"}', call_id='call_jjczJmrW70WMyD1IpwAaeW40', name='search', type='function_call', id='fc_0b59cd8ff27fb8a2006a40369592188197bd98064fdba5992e', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_jjczJmrW70WMyD1IpwAaeW40', 'outp

In [76]:
result2.cost


CostInfo(input_cost=Decimal('0.001896'), output_cost=Decimal('0.0011835'), total_cost=Decimal('0.0030795'))

Type questions and get answers. Type "stop" to exit.
